# NB5b (1/3) — inputs for the foundation models (development)

D23, D59; guide §5.4 rows A6a–A6c. The 10 validation units and the issue times NB5a forecast. For each unit: the event's index k, the fleet state (σ, c, n) and the sun geometry u on the 15-min grid, from the fleet's start (2021-05-15, LightGBM's first training day) to two days after the last issue time. Writes `results/nb5b_inputs/` (series.parquet and tasks.parquet, git-ignored; inputs_manifest.json with their sha256, tracked). The foundation models read only these files.

In [1]:
import os, sys, time, json, hashlib
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT / 'src')); sys.path.insert(0, str(ROOT / 'tools'))
from pvcold import config as C, audit as A, forecast as FC
import r1_score as R                      # NB5a context and slots (D37)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
t0 = time.time()
ctx = R.load_context()
VAL, fc5, idx, geo15, E = ctx['VAL'], ctx['fc5'], ctx['idx'], ctx['geo15'], ctx['E']
COLS = ['minute', 'k', 'sigma', 'c', 'n', 'Gt', 'cosz', 'sin_az', 'cos_az', 'sin_ha', 'cos_ha', 'sin_doy', 'cos_doy']
tasks = fc5[['event', 'origin', 't']].drop_duplicates().sort_values(['event', 't']).reset_index(drop=True)
assert set(tasks.event) == set(VAL) and all(E.loc[v, 'set'] == 'validation' for v in VAL)
parts, chk = [], []
for v in VAL:
    S = FC.event_series(v, idx, geo15)
    t_last = tasks.loc[tasks.event == v, 't'].max()
    S = S.loc[FC.TRAIN_START:t_last.normalize() + pd.Timedelta(days=2) - pd.Timedelta(minutes=15)]
    day = (S['minute'] >= 360) & (S['minute'] <= FC.LAST_TARGET_MIN)
    chk.append({'event': v, 'rows': len(S), 'k_outside_daylight': int(np.isfinite(S.loc[~day, 'k']).sum()),
                'Gt_missing_in_daylight': int(S.loc[day, 'Gt'].isna().sum()), 'tasks': int((tasks.event == v).sum())})
    out = S[COLS].copy(); out.insert(0, 'time', S.index); out.insert(0, 'event', v)
    parts.append(out.reset_index(drop=True))
series = pd.concat(parts, ignore_index=True)
chk = pd.DataFrame(chk); print(chk.to_string(index=False))
assert (chk.k_outside_daylight == 0).all() and (chk.Gt_missing_in_daylight == 0).all()
print('series', series.shape, '| tasks', len(tasks))

            event  rows  k_outside_daylight  Gt_missing_in_daylight  tasks
            SQ567 10464                   0                       0   3035
      UG Hall2 2F 10560                   0                       0   3033
      UG Hall2 RF 11136                   0                       0   3904
          Zone A1 11328                   0                       0   3901
          Zone A4 11328                   0                       0   3901
        Zone A2 1 13248                   0                       0   3988
 SQ Apartment1-12 13920                   0                       0   3974
SQ Apartment13-24 14208                   0                       0   3966
SQ Apartment25-36 14592                   0                       0   3951
SQ Apartment37-48 14208                   0                       0   3966
series (124992, 15) | tasks 37619


In [2]:
OUT = C.RESULTS / 'nb5b_inputs'; TMP = C.RESULTS / '.tmp_nb5b_inputs'
import shutil
if TMP.exists(): shutil.rmtree(TMP)
TMP.mkdir(parents=True)
series.to_parquet(TMP / 'series.parquet'); tasks.to_parquet(TMP / 'tasks.parquet')
man = {'notebook': 'NB5b_inputs', 'environment': A.versions(), 'runtime_s': round(time.time() - t0, 1),
       'events': VAL, 'n_series_rows': len(series), 'n_tasks': len(tasks),
       'sha256': {f: hashlib.sha256((TMP / f).read_bytes()).hexdigest() for f in ['series.parquet', 'tasks.parquet']}}
(TMP / 'inputs_manifest.json').write_bytes(json.dumps(man, indent=2, default=str).encode('utf-8'))
if OUT.exists(): shutil.rmtree(OUT)
TMP.rename(OUT)
print(json.dumps(man['sha256'], indent=1)); print('written to', OUT)

{
 "series.parquet": "35af0c3f78bdd0a26f1e93b1a8d143fd9123e8d734ca66189aeea30b484ef9a9",
 "tasks.parquet": "e6f3429c9578b66dd0d3d58724ef1509b16238265e9c7e6ed49ca7257b491020"
}
written to C:\Users\Dell\Desktop\phd_ehtp\pv paper\results\nb5b_inputs
